In [0]:
%python
dbutils.widgets.removeAll()

In [0]:
create widget text storageName default "adlsantodata1703";

In [0]:
%python
storageName = dbutils.widgets.get("storageName")

In [0]:
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-metastore`
URL 'abfss://metastore@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas raw del Data Lake';

In [0]:
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-raw`
URL 'abfss://raw@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas raw del Data Lake';

In [0]:
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-bronze`
URL 'abfss://bronze@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas bronze del Data Lake';

In [0]:
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-silver`
URL 'abfss://silver@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas silver del Data Lake';

In [0]:
CREATE EXTERNAL LOCATION IF NOT EXISTS `exlt-golden`
URL 'abfss://golden@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas golden del Data Lake';

In [0]:
DROP CATALOG IF EXISTS catalog_au CASCADE;

In [0]:
CREATE CATALOG IF NOT EXISTS catalog_au
MANAGED LOCATION 'abfss://metastore@${storageName}.dfs.core.windows.net/'
COMMENT 'Catalogo para la arquitectura medallion del ambiente de dev';

In [0]:
DROP SCHEMA IF EXISTS catalog_au.raw;
DROP SCHEMA IF EXISTS catalog_au.bronze;
DROP SCHEMA IF EXISTS catalog_au.silver;
DROP SCHEMA IF EXISTS catalog_au.golden;

In [0]:
%python
dbutils.fs.rm(f"abfss://bronze@{storageName}.dfs.core.windows.net/",True)
dbutils.fs.rm(f"abfss://silver@{storageName}.dfs.core.windows.net/",True)
dbutils.fs.rm(f"abfss://golden@{storageName}.dfs.core.windows.net/",True)

In [0]:
CREATE SCHEMA IF NOT EXISTS catalog_au.raw;
CREATE SCHEMA IF NOT EXISTS catalog_au.bronze;
CREATE SCHEMA IF NOT EXISTS catalog_au.silver;
CREATE SCHEMA IF NOT EXISTS catalog_au.golden;

###Tablas Bronze

In [0]:
CREATE TABLE IF NOT EXISTS catalog_au.bronze.circuits (
circuit_id integer,
circuit_ref string,
name string,
location string,
country string,
latitude double,
longitude double,
altitude integer,
ingestion_date timestamp
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/circuits"

In [0]:
CREATE TABLE IF NOT EXISTS catalog_au.bronze.races (
race_id integer,
race_year integer,
round integer,
circuit_id integer,
name string,
ingestion_date timestamp
)
USING DELTA
PARTITIONED BY (race_year)
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/races"

###Tablas Silver

In [0]:
CREATE TABLE IF NOT EXISTS catalog_au.silver.circuits_transformed (
  race_id integer,
  race_year integer,
  round integer,
  circuit_id integer,
  name_race string,
  ingestion_date timestamp,
  circuit_ref string,
  name string,
  location string,
  country string,
  latitude double,
  longitude double,
  altitude integer,
  altitude_category string,
  years_diferences integer,
  lat_diff integer,
  race_type string,
  near_equator string
)
USING DELTA
LOCATION "abfss://silver@${storageName}.dfs.core.windows.net/circuits_transformed"

###Tablas Golden

In [0]:
CREATE TABLE IF NOT EXISTS catalog_au.golden.golden_raced_partitioned (
  race_year integer,
  conteo long,
  max_altitude integer,
  min_altitude integer,
  country string,
  race_type string,
  near_equator string
)
USING DELTA
LOCATION "abfss://golden@${storageName}.dfs.core.windows.net/golden_raced_partitioned"